# ADWIN: Adaptive Windowing for Drift Detection

Wiki reference for [ADWIN](https://ml-viz-ruby.vercel.app/wiki/adwin).

**The idea in one sentence.** Keep a window of recent values and drop the stale older half whenever two sub-windows' means differ by more than a Hoeffding cut — detecting drift and rightsizing the window at once.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## 1. From scratch — the split test and the Hoeffding cut

For a window `W`, test splits into an older `W0` and newer `W1`; cut when `|μ̂_W0 − μ̂_W1| > ε_cut`, with `ε_cut = √((1/2m)·ln(4|W|/δ))` and `m` the harmonic mean of the sub-window sizes. On a cut, keep only `W1` and report drift.

In [ ]:
def epsilon_cut(n0, n1, W_len, delta):
    m = 1.0 / (1.0 / n0 + 1.0 / n1)          # harmonic mean of sub-window sizes
    return np.sqrt((1.0 / (2 * m)) * np.log(4 * W_len / delta))

def adwin_detect(stream, delta=0.05, min_sub=30):
    start = 0
    for t in range(min_sub, len(stream)):
        win = stream[start:t + 1]
        L = len(win)
        pre = np.cumsum(win)
        for s in range(min_sub, L - min_sub):
            m0 = pre[s - 1] / s
            m1 = (pre[-1] - pre[s - 1]) / (L - s)
            if abs(m0 - m1) > epsilon_cut(s, L - s, L, delta):
                return t          # detection time
    return None

### Try it on a stream with a concept shift

Build an error stream that is low-error before `D` and high-error after, then run the detector.

In [ ]:
D = 1000
stream = np.concatenate([
    (np.random.rand(D) < 0.08).astype(int),        # low-error concept
    (np.random.rand(1000) < 0.42).astype(int),     # concept shifts -> higher error
])
ours = adwin_detect(stream, delta=0.05)
print(f'from-scratch ADWIN detected at t={ours}  (delay {ours - D})')
assert ours is not None and D <= ours <= D + 200, 'should detect shortly after the shift'

## 2. The library way — validate against river's ADWIN

`river.drift.ADWIN` is the production detector. We feed it the same error stream and check it also fires shortly after `D`. Installs river if needed, with a graceful offline fallback.

In [ ]:
try:
    from river import drift
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'river'], check=False)
    try:
        from river import drift
    except ImportError:
        drift = None

if drift is not None:
    adwin = drift.ADWIN(delta=0.002)
    river_det = None
    for i, b in enumerate(stream):
        adwin.update(int(b))
        if adwin.drift_detected:
            river_det = i; break
    print(f'river ADWIN detected at t={river_det}')
    assert river_det is not None and D <= river_det <= D + 300
    print('from-scratch and river ADWIN both fire shortly after the shift ✓')
else:
    print('river unavailable offline — from-scratch detector already validated above')

## 3. Worked cut calculation — reproduce the wiki numbers

A balanced split of a `|W| = 200` window: older mean 0.08, newer 0.42, `δ = 0.05`.

In [ ]:
eps = epsilon_cut(100, 100, 200, 0.05)
gap = abs(0.08 - 0.42)
print(f'epsilon_cut = {eps:.3f}   observed gap = {gap:.3f}   -> cut fires: {gap > eps}')
assert abs(eps - 0.311) < 0.005, 'matches the worked example in the wiki'
assert gap > eps

**What to notice:** the gap `0.34` exceeds the cut `0.311`, so ADWIN drops the older half and reports drift — the exact calculation from the wiki page. As the window grows, `m` rises and `ε_cut` falls, so subtler changes become detectable once more post-change data accumulates.

## 4. Visualize it — detection delay vs confidence δ

Sweeping `δ` traces the universal trade-off: larger `δ` lowers `ε_cut`, catching drift sooner.

In [ ]:
deltas = [0.005, 0.02, 0.05, 0.1, 0.3]
delays = [adwin_detect(stream, delta=d) for d in deltas]
delays = [None if d is None else d - D for d in delays]
print('delta -> detection delay:', dict(zip(deltas, delays)))
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(deltas, delays, 'o-', color='#14b8a6')
ax.set_xscale('log'); ax.set_xlabel('confidence δ (log)'); ax.set_ylabel('detection delay')
ax.set_title('Larger δ → faster detection (and more false alarms)', color='white')
ax.grid(alpha=0.2, which='both'); plt.show()

**What to notice:** detection delay falls as `δ` rises — but the same lower threshold that catches real drift sooner also fires more readily on ordinary noise, so very large `δ` risks false alarms on stationary data. Choose `δ` for your tolerance to spurious resets.

## 5. Your turn

### Exercise — the Hoeffding cut

Implement `ε_cut = √( (1/2m)·ln(4|W|/δ) )` where `m` is the harmonic mean of the two sub-window sizes `n0, n1`.

In [ ]:
def eps_cut(n0, n1, W_len, delta):
    # TODO(you): m = harmonic mean of n0, n1; return sqrt((1/2m) * ln(4*W_len/delta))
    return ...


In [ ]:
# Checks — run me
assert abs(eps_cut(100, 100, 200, 0.05) - 0.311) < 0.005, 'the worked example value'
# a larger delta lowers the threshold (easier to trip)
assert eps_cut(100, 100, 200, 0.3) < eps_cut(100, 100, 200, 0.05)
# an unbalanced split (small side) inflates the threshold
assert eps_cut(5, 195, 200, 0.05) > eps_cut(100, 100, 200, 0.05)
print('✅ Exercise passed')

<details>
<summary>💡 Show solution</summary>

```python
def eps_cut(n0, n1, W_len, delta):
    m = 1.0 / (1.0 / n0 + 1.0 / n1)
    return np.sqrt((1.0 / (2 * m)) * np.log(4 * W_len / delta))
```

</details>

## 6. Key takeaways

- ADWIN cuts its window when two halves' means differ by more than the **Hoeffding cut** `ε_cut`.
- The retained window length **rightsizes itself** to the current concept — no window hyperparameter.
- An **exponential histogram** (DGIM-style buckets) gives `O(log W)` memory and split points.
- `δ` trades **detection delay vs false alarms**.
- Back to [Concept Drift & Adaptation](https://ml-viz-ruby.vercel.app/courses/streaming-ml/04-concept-drift).